### The environment 🎮

- [LunarLander-v2](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

### The library used 📚

- [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/)

We're constantly trying to improve our tutorials, so **if you find some issues in this notebook**, please [open an issue on the Github Repo](https://github.com/huggingface/deep-rl-class/issues).

## Objectives of this notebook 🏆

At the end of the notebook, you will:

- Be able to use **Gymnasium**, the environment library.
- Be able to use **Stable-Baselines3**, the deep reinforcement learning library.
- Be able to **push your trained agent to the Hub** with a nice video replay and an evaluation score 🔥.




In [ ]:
# Installing all the dependencies
!apt install swig cmake

In [ ]:
# Dependencies - 2
#!pip install -r https://raw.githubusercontent.com/huggingface/deep-rl-class/main/notebooks/unit1/requirements-unit1.txt

# Install the current versions of all libraries (replaces the outdated requirements file)
!pip install "gymnasium[box2d]" stable-baselines3 huggingface_sb3 shimmy

In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

To make sure the new installed libraries are used, **sometimes it's required to restart the notebook runtime**. The next cell will force the **runtime to crash, so you'll need to connect again and run the code starting from here**. Thanks to this trick, **we will be able to run our virtual screen.**

In [ ]:
#import os
#os.kill(os.getpid(), 9)

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [ ]:
# Import packages
import gymnasium

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login # To log to our Hugging Face account to be able to upload models to the Hub.

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

In [ ]:
#Testing the env
import gymnasium as gym

# Create the LunarLander game (v3 is the current version of v2)
env = gym.make("LunarLander-v3")

# Reset the game to its starting state
observation, info = env.reset()

for _ in range(20):
    # Pick a random action (0: nothing, 1: left engine, 2: main engine, 3: right engine)
    action = env.action_space.sample()
    print("Action taken:", action)

    # Do the action and get the result: new state, reward, and whether the episode ended
    observation, reward, terminated, truncated, info = env.step(action)

    # If the lander landed, crashed, or timed out, start a new episode
    if terminated or truncated:
        print("Environment is reset")
        observation, info = env.reset()

env.close()

Let's see what the Environment looks like:


In [ ]:
# Creatign the environment and the PPO model
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# Run 16 copies of the game at once, so the agent gets more varied experience per step
env = make_vec_env("LunarLander-v3", n_envs=16)

# Create the PPO agent (the "brain" that will learn to land)
model = PPO(
    policy="MlpPolicy",  # a standard neural network, used because the game state is just 8 numbers
    env=env,
    n_steps=1024,        # steps each game copy plays before the agent learns from them
    batch_size=64,       # how much data is used per learning update
    n_epochs=4,          # how many times it reuses each batch of experience
    gamma=0.999,         # how much it values future rewards (close to 1 = plans ahead)
    gae_lambda=0.98,     # smooths how it estimates which actions were good
    ent_coef=0.01,       # small bonus for trying new actions, so it keeps exploring
    verbose=1,           # print training progress
)

In [ ]:
# Recreating the model on CPU and training again
from stable_baselines3 import PPO

# Same PPO agent as before, but forced onto the CPU, which is faster for this kind of network
model = PPO(
    policy="MlpPolicy",
    env=env,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    verbose=1,
    device="cpu",
)

# Train for 1,000,000 steps: the agent plays, gets rewards, and improves its policy
model.learn(total_timesteps=1000000)

# Save the trained agent to a file
model_name = "ppo-LunarLander-v3"
model.save(model_name)

In [ ]:
# Evaluate the agent
import gymnasium as gym
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy

# Create a fresh game just for testing (not the training one)
eval_env = Monitor(gym.make("LunarLander-v3"))

# Play 10 test episodes with no random exploration and record the scores
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)

print(f"mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")
print(f"Leaderboard score (mean - std) = {mean_reward - std_reward:.2f}")

In [ ]:
from huggingface_hub import notebook_login

# Log in to Hugging Face so Colab is allowed to upload to your account
notebook_login()

In [ ]:
import gymnasium as gym
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from huggingface_sb3 import package_to_hub

env_id = "LunarLander-v3"
model_architecture = "PPO"
repo_id = "Yoga-Varshitha-Jajula/ppo-LunarLander-v3"
commit_message = "Upload PPO LunarLander-v3 trained agent"

# Evaluation environment that can record a video of your agent
eval_env = DummyVecEnv([lambda: Monitor(gym.make(env_id, render_mode="rgb_array"))])

# Evaluates the agent, records a replay video, writes a model card, and uploads everything
package_to_hub(
    model=model,
    model_name=model_name,
    model_architecture=model_architecture,
    env_id=env_id,
    eval_env=eval_env,
    repo_id=repo_id,
    commit_message=commit_message,
)

In [ ]:
# Older Gymnasium that still includes LunarLander-v2 (the checker only recognises v2)
!pip install -q "gymnasium==0.29.1" box2d pygame-ce

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv
from huggingface_sb3 import package_to_hub
from huggingface_hub import notebook_login

# Log in again after the restart (paste your write token if asked)
notebook_login()

# Load the agent you already trained (no retraining needed)
model = PPO.load("ppo-LunarLander-v3.zip", device="cpu")

# Test it on LunarLander-v2, the version the progress checker requires
eval_env = Monitor(gym.make("LunarLander-v2"))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
score = mean_reward - std_reward
print(f"mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")
print(f"Leaderboard score (mean - std) = {score:.2f}")

# Push to a new v2 repo only if the score passes
if score >= 200:
    package_to_hub(
        model=model,
        model_name="ppo-LunarLander-v2",
        model_architecture="PPO",
        env_id="LunarLander-v2",
        eval_env=DummyVecEnv([lambda: Monitor(gym.make("LunarLander-v2", render_mode="rgb_array"))]),
        repo_id="Yoga-Varshitha-Jajula/ppo-LunarLander-v2",
        commit_message="Upload PPO LunarLander-v2 trained agent",
    )
else:
    print("Score is below 200, so nothing was pushed. Send me these numbers.")